# AGH-Former Ortodontik Landmark - Colab GPU

Bu notebook, önerilen **Anatomy-Aware Geodesic Heatmap Transformer (AGH-Former)** modelini 23 landmarklı 3B ortodontik yüz dataseti üzerinde çalıştırmak için hazırlanmıştır.

Varsayılan veri yapısı:

```text
CODE_ROOT     = /content/comparative-study
DATA_ROOT     = /content/drive/MyDrive/orthodontic/data/dataset
SPLITS_JSON   = /content/comparative-study/shared_splits/orthodontic_180_60_60_seed42.json
TRANSFORM_DIR = /content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801
RUN_ROOT      = /content/drive/MyDrive/orthodontic/diffusion_runs
```


In [ ]:
import os, platform, torch
print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 1. Drive Bağlantısı

Dataset ve transform dosyaları GitHub reposunda olmadığı için Google Drive'dan okunur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Repo Hazırlığı

Repo `/content/comparative-study` altında yoksa GitHub'dan klonlanır. Zaten varsa son hali çekilir.

In [ ]:
from pathlib import Path
import subprocess, os

CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'

if not CODE_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)

print('CODE_ROOT:', CODE_ROOT)


## 3. Yol Kontrolü

Bu hücre dataset, split ve transform yollarını kontrol eder. `TRANSFORM_DIR` yoksa model raw koordinatlarla çalışabilir; ancak adil karşılaştırma için Procrustes transformlarının bulunması önerilir.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/orthodontic/data/dataset')
SPLITS_JSON = Path('/content/comparative-study/shared_splits/orthodontic_180_60_60_seed42.json')
TRANSFORM_DIR = Path('/content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801')
RUN_ROOT = Path('/content/drive/MyDrive/orthodontic/diffusion_runs')
USE_TRANSFORMS = TRANSFORM_DIR.exists()

print('DATA_ROOT:', DATA_ROOT, DATA_ROOT.exists())
print('SPLITS_JSON:', SPLITS_JSON, SPLITS_JSON.exists())
print('TRANSFORM_DIR:', TRANSFORM_DIR, TRANSFORM_DIR.exists())
print('RUN_ROOT:', RUN_ROOT)
print('USE_TRANSFORMS:', USE_TRANSFORMS)

print('PLY count:', len(list(DATA_ROOT.glob('Class*/men/*.ply'))) + len(list(DATA_ROOT.glob('Class*/women/*.ply'))))
print('Landmark TXT count:', len(list(DATA_ROOT.glob('Class*/Class*-Landmark/men/*.txt'))) + len(list(DATA_ROOT.glob('Class*/Class*-Landmark/women/*.txt'))))
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## 4. Kurulum

AGH-Former bağımlılıkları yüklenir.

In [ ]:
%cd /content/comparative-study
!pip install -q -r agh_former_orthodontic_comparison/requirements.txt


## 5. Smoke Test

Bu test yalnızca kod hattının çalıştığını doğrular. Performans sonucu bilimsel değerlendirme için kullanılmaz.

In [ ]:
%cd /content/comparative-study/agh_former_orthodontic_comparison
!python -u colab_run_aghformer_shared_metrics.py --preset smoke


## 6. A100 Ana Koşu

Bu hücre ana AGH-Former eğitimini başlatır. Uzun sürebilir. Çıktılar `RUN_ROOT` altında saklanır.

In [ ]:
%cd /content/comparative-study/agh_former_orthodontic_comparison
!python -u colab_run_aghformer_shared_metrics.py --preset a100


## 7. Büyük Bellek Koşusu

A100 veya yüksek VRAM ortamında daha yoğun yüzey örnekleme için kullanılır.

In [ ]:
%cd /content/comparative-study/agh_former_orthodontic_comparison
!python -u colab_run_aghformer_shared_metrics.py --preset a100_16k


## 8. Sonuçları Oku

Tamamlanan koşunun `metrics.json` dosyasından ana metrikleri özetler.

In [ ]:
import json
from pathlib import Path

run_dir = Path('/content/drive/MyDrive/orthodontic/diffusion_runs/aghformer_v2_template_p12000_w192_b4_e220')
metrics_path = run_dir / 'metrics.json'
print('metrics:', metrics_path, metrics_path.exists())
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print('AGH-Former snapped ALE:', metrics['aghformer_snapped']['ale'])
    print('AGH-Former snapped median:', metrics['aghformer_snapped']['median'])
    print('PCK@2mm:', metrics['aghformer_snapped'].get('pck_at_2mm'))
    print('PCK@2.5mm:', metrics['aghformer_snapped'].get('pck_at_2_5mm'))
    print('PCK@3mm:', metrics['aghformer_snapped'].get('pck_at_3mm'))


## 9. Evaluate-only

Eğitilmiş `best_model.pth` üzerinden yeniden test değerlendirmesi yapar.

In [ ]:
%cd /content/comparative-study/agh_former_orthodontic_comparison
!python -u run_orthodontic_aghformer.py \
  --data-root /content/drive/MyDrive/orthodontic/data/dataset \
  --splits-json /content/comparative-study/shared_splits/orthodontic_180_60_60_seed42.json \
  --transformation-dir /content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801 \
  --output-dir /content/drive/MyDrive/orthodontic/diffusion_runs/aghformer_v2_template_p12000_w192_b4_e220 \
  --surface-points 12000 \
  --width 192 \
  --blocks 4 \
  --heads 6 \
  --evaluate-only \
  --model-path /content/drive/MyDrive/orthodontic/diffusion_runs/aghformer_v2_template_p12000_w192_b4_e220/best_model.pth \
  --template-mode class_gender \
  --prediction-mode direct \
  --selection-metric raw \
  --topk 30 \
  --device auto
